# Welcome to RAG week!!

## Expert Knowledge Worker

### A question answering Assistant that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The AI assistant needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

This first implementation will use a simplistic, brute-force type of RAG..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications of this week's projects</h2>
            <span style="color:#181;">RAG is perhaps the most immediately applicable technique of anything that we cover in the course! In fact, there are commercial products that do precisely what we build this week: nuanced querying across large databases of information, such as company contracts or product specs. RAG gives you a quick-to-market, low cost mechanism for adapting an LLM to your business area.</span>
        </td>
    </tr>
</table>

In [1]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            This lab, and all the labs for Week 5, has been updated to use LangChain 1.0. This is intended to be reviewed with the new video series as of November 2025. If you're reviewing the older video series, then please consider doing <code>git checkout original</code> to revert to the prior code, then later <code>git checkout main</code> to get back to the new code. I have a really exciting week ahead, with evals and Advanced RAG!
            </span>
        </td>
    </tr>
</table>

In [2]:
# Setting up

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


### Let's read in all employee data into a dictionary

In [67]:
knowledge = {}

filenames = glob.glob("DALL_FULL_MD_Dataset/employees/*")
print(filenames)
for filename in filenames:
    print(Path(filename).stem)
    name = Path(filename).stem.split(' ')[-1]
    print(name)
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

['DALL_FULL_MD_Dataset/employees\\emp_001 john doe.md', 'DALL_FULL_MD_Dataset/employees\\emp_002 priya sharma.md', 'DALL_FULL_MD_Dataset/employees\\emp_003 rahul verma.md', 'DALL_FULL_MD_Dataset/employees\\emp_004 sara khan.md', 'DALL_FULL_MD_Dataset/employees\\emp_005 amit patnaik.md', 'DALL_FULL_MD_Dataset/employees\\emp_006 neha singh.md', 'DALL_FULL_MD_Dataset/employees\\emp_007 rohit mehta.md', 'DALL_FULL_MD_Dataset/employees\\emp_008 ananya roy.md', 'DALL_FULL_MD_Dataset/employees\\emp_009 kunal jain.md', 'DALL_FULL_MD_Dataset/employees\\emp_010 meera nair.md']
emp_001 john doe
doe
emp_002 priya sharma
sharma
emp_003 rahul verma
verma
emp_004 sara khan
khan
emp_005 amit patnaik
patnaik
emp_006 neha singh
singh
emp_007 rohit mehta
mehta
emp_008 ananya roy
roy
emp_009 kunal jain
jain
emp_010 meera nair
nair


In [68]:
knowledge

{'doe': '# Employee Profile: John Doe\nRole: Senior Hardware Engineer\nSkills: Server design, DDR5, Signal integrity, Thermal optimization\nCareer: Joined DALL in 2019, promoted to Senior Engineer in 2022\n',
 'sharma': '# Employee Profile: Priya Sharma\nRole: Memory Systems Engineer\nSkills: DDR5, HBM, Firmware validation, Python testing\nCareer: Joined in 2020, key contributor to HBM stack\n',
 'verma': '# Employee Profile: Rahul Verma\nRole: Data Center Operations Manager\nSkills: SLA management, Cooling optimization, Capacity planning\nCareer: Joined in 2017, manages multi-region DCs\n',
 'khan': '# Employee Profile: Sara Khan\nRole: QA Lead\nSkills: Hardware testing, Stress testing, Compliance\nCareer: Joined in 2018, leads QA automation\n',
 'patnaik': '# Employee Profile: Amit Patnaik\nRole: Supply Chain Manager\nSkills: Vendor management, ERP, Logistics\nCareer: Joined in 2016, optimized supply cost\n',
 'singh': '# Employee Profile: Neha Singh\nRole: Firmware Engineer\nSkills:

In [69]:
knowledge["doe"]

'# Employee Profile: John Doe\nRole: Senior Hardware Engineer\nSkills: Server design, DDR5, Signal integrity, Thermal optimization\nCareer: Joined DALL in 2019, promoted to Senior Engineer in 2022\n'

In [70]:
filenames = glob.glob("DALL_FULL_MD_Dataset/products/*")

print(filenames)
for filename in filenames:
    print(Path(filename).stem)
    name = Path(filename).stem.split(' ')[-1]
    print(name)
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

['DALL_FULL_MD_Dataset/products\\ai server.md', 'DALL_FULL_MD_Dataset/products\\controller.md', 'DALL_FULL_MD_Dataset/products\\ddr5_64gb.md', 'DALL_FULL_MD_Dataset/products\\edge_e50.md', 'DALL_FULL_MD_Dataset/products\\hbm_stack.md', 'DALL_FULL_MD_Dataset/products\\micro server.md', 'DALL_FULL_MD_Dataset/products\\nvme_array.md', 'DALL_FULL_MD_Dataset/products\\rack_r200.md', 'DALL_FULL_MD_Dataset/products\\secure server.md', 'DALL_FULL_MD_Dataset/products\\server x100.md']
ai server
server
controller
controller
ddr5_64gb
ddr5_64gb
edge_e50
edge_e50
hbm_stack
hbm_stack
micro server
server
nvme_array
nvme_array
rack_r200
rack_r200
secure server
server
server x100
x100


In [72]:
filenames = glob.glob("DALL_FULL_MD_Dataset/contracts/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [73]:
filenames = glob.glob("DALL_FULL_MD_Dataset/company/*")

for filename in filenames:
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [74]:
knowledge.keys()

dict_keys(['doe', 'sharma', 'verma', 'khan', 'patnaik', 'singh', 'mehta', 'roy', 'jain', 'nair', 'server', 'controller', 'ddr5_64gb', 'edge_e50', 'hbm_stack', 'nvme_array', 'rack_r200', 'x100', 'contract_001_enterprise', 'contract_002_government', 'contract_003_datacenter', 'contract_004_maintenance', 'contract_005_cloud', 'contact'])

In [75]:
SYSTEM_PREFIX = """
You represent DALLLLM, the Product Tech company.
You are an expert in answering questions about DALLLLM; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [76]:
def get_relevant_context_simple(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    print(f"Cleaned text: {text}")
    words = text.lower().split()
    print(f"Words: {words}")
    relevant_context = []
    for word in words:
        if word in knowledge:
            relevant_context.append(knowledge[word])
    return relevant_context          

In [56]:
get_relevant_context_simple("Tell me about John Doe and the SuperWidget product !. ?")

Cleaned text: Tell me about John Doe and the SuperWidget product  
Words: ['tell', 'me', 'about', 'john', 'doe', 'and', 'the', 'superwidget', 'product']


['# Employee Profile: John Doe\nRole: Senior Hardware Engineer\nSkills: Server design, DDR5, Signal integrity, Thermal optimization\nCareer: Joined DALL in 2019, promoted to Senior Engineer in 2022\n']

## But a more pythonic way:

In [77]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]   

In [78]:
get_relevant_context("Who is khan?")

['# Employee Profile: Sara Khan\nRole: QA Lead\nSkills: Hardware testing, Stress testing, Compliance\nCareer: Joined in 2018, leads QA automation\n']

In [80]:
get_relevant_context("AI Compute Server")

['Secure Server\nPrice: ₹8.4L–₹11.6L\nFeatures: TPM, Secure Boot\n']

In [81]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [83]:
print(additional_context("Supply Contract"))

There is no additional context relevant to the user's question.


In [84]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## Now we will bring this up in Gradio using the Chat interface -

A quick and easy way to prototype a chat with an LLM

In [85]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
